<a href="https://colab.research.google.com/github/dmsejn4/TSD-sequential-adaptation/blob/main/ext_exp_for_val.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Heqiang-Huang/YOLO-TS.git
%cd /content/YOLO-TS

Cloning into 'YOLO-TS'...
remote: Enumerating objects: 728, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 728 (delta 59), reused 22 (delta 22), pack-reused 636 (from 1)
Receiving objects: 100% (728/728), 10.65 MiB | 28.40 MiB/s, done.
Resolving deltas: 100% (331/331), done.
/content/YOLO-TS


In [ ]:
!pip install -r requirements.txt

### mount

In [2]:
from google.colab import drive
# drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


### data load

In [3]:
# 1. 경로 이동
%cd /content/drive/MyDrive/CCTSDB2021

# 2. 기존 dataset 폴더 완전 초기화
!rm -rf /content/dataset

# 3. 디렉토리 재생성
!mkdir -p /content/dataset/images/train
!mkdir -p /content/dataset/images/val
!mkdir -p /content/dataset/images/test
!mkdir -p /content/dataset/labels/train
!mkdir -p /content/dataset/labels/val
!mkdir -p /content/dataset/labels/test

# 4. 압축 해제 (-j: 폴더 구조 무시하고 파일만 평탄하게 추출)
!unzip -q -j train_img.zip -d /content/dataset/images/train
!unzip -q -j test_img.zip -d /content/dataset/images/test
!unzip -q -j train_labels.zip -d /content/dataset/labels/train
!unzip -q -j test_labels.zip -d /content/dataset/labels/test

/content/drive/.shortcut-targets-by-id/14Km2W-5hbixXDfz7WSqW_Rx7O5m8ZMFn/CCTSDB2021


In [4]:
# import random
# import shutil
# from pathlib import Path

# random.seed(0)  # 재현성을 위한 시드 고정

# img_dir = Path('/content/dataset/images/train')
# label_dir = Path('/content/dataset/labels/train')
# val_img_dir = Path('/content/dataset/images/val')
# val_label_dir = Path('/content/dataset/labels/val')

# # split.txt는 Drive에 저장 -> /content/dataset은 cell 13에서 매번 초기화되므로
# # 여기 저장하면 다음 실행 때 사라짐
# split_txt_path = Path('/content/drive/MyDrive/CCTSDB2021/split.txt')

# img_exts = ('.jpg', '.jpeg', '.png')

# if split_txt_path.exists():
#     # 이전에 저장된 분할을 그대로 재현 (매 실행 동일 조건 보장)
#     val_names = [
#         line.split('\t', 1)[1] for line in split_txt_path.read_text(encoding='utf-8').splitlines()
#         if line.startswith('val\t')
#     ]
#     val_files = [img_dir / name for name in val_names]
#     print(f"기존 split.txt 발견 → 저장된 분할을 재사용합니다 (val {len(val_files)}개)")
# else:
#     # iterdir()의 반환 순서는 실행마다 달라질 수 있어 seed를 고정해도 결과가 흔들렸음
#     # -> 정렬로 입력 순서를 고정한 뒤 셔플
#     img_files = sorted(f for f in img_dir.iterdir() if f.suffix.lower() in img_exts)
#     print(f"전체 train 이미지 개수: {len(img_files)}")

#     random.shuffle(img_files)
#     val_count = int(len(img_files) * 0.2)
#     val_files = img_files[:val_count]
#     print(f"val로 이동할 개수: {len(val_files)} (새로 분할 - split.txt 없음)")

# moved, missing_label = 0, 0
# for img_path in val_files:
#     label_path = label_dir / (img_path.stem + '.txt')

#     # 이미지 이동
#     shutil.move(str(img_path), str(val_img_dir / img_path.name))

#     # 라벨 이동 (혹시 라벨 없는 이미지 대비)
#     if label_path.exists():
#         shutil.move(str(label_path), str(val_label_dir / label_path.name))
#         moved += 1
#     else:
#         missing_label += 1
#         print(f"라벨 없음: {label_path.name}")

# print(f"이동 완료: 이미지 {len(val_files)}개, 라벨 {moved}개 (라벨 누락 {missing_label}개)")
# print(f"남은 train 이미지: {len(list(img_dir.glob('*')))}")
# print(f"val 이미지: {len(list(val_img_dir.glob('*')))}")

전체 train 이미지 개수: 16356
val로 이동할 개수: 3271 (새로 분할 - split.txt 없음)
이동 완료: 이미지 3271개, 라벨 3271개 (라벨 누락 0개)
남은 train 이미지: 13085
val 이미지: 3271


In [6]:
# from pathlib import Path
# WORK = Path('/content/drive/MyDrive/review2026')
# WORK.mkdir(parents=True, exist_ok=True)

# val_dir = Path('/content/dataset/images/val')
# names = sorted(f.name for f in val_dir.iterdir() if f.suffix.lower() in ('.jpg','.jpeg','.png'))
# (WORK/'split_cctsdb2021.txt').write_text('\n'.join(f'val\t{n}' for n in names), encoding='utf-8')
# print(f"저장 완료: {len(names)}개")

저장 완료: 3271개


In [8]:
from pathlib import Path
print(sum(len([l for l in f.read_text().splitlines() if l.strip()])
          for f in Path('/content/dataset/labels/val').glob('*.txt'))) #5434

5434


### 이다음부터 split 고정값 사용

In [ ]:
import random, shutil
from pathlib import Path

# ── 데이터셋별 설정 ────────────────────────────
WORK = Path('/content/drive/MyDrive/review2026')
SPLIT_TXT = WORK / 'split_cctsdb2021.txt'
EXPECTED_TRAIN = 16356      # TT100K: 6103, Gen-TT: 6105
EXPECTED_VAL_LINES = 5434   # 최초 1회 확인 후 기록
VAL_RATIO, SEED = 0.2, 0
# ──────────────────────────────────────────────

ROOT = Path('/content/dataset')
img_dir, label_dir = ROOT/'images/train', ROOT/'labels/train'
val_img_dir, val_label_dir = ROOT/'images/val', ROOT/'labels/val'
IMG_EXTS = ('.jpg', '.jpeg', '.png')

img_files = sorted(f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS)
assert len(img_files) == EXPECTED_TRAIN, \
    f"원본 상태가 아닙니다. 기대 {EXPECTED_TRAIN}, 실제 {len(img_files)}. zip 해제부터 다시 하세요."

if SPLIT_TXT.exists():
    val_names = [line.split('\t', 1)[1]
                 for line in SPLIT_TXT.read_text(encoding='utf-8').splitlines()
                 if line.startswith('val\t')]
    print(f"기존 split 재사용: val {len(val_names)}개")
else:
    rng = random.Random(SEED)
    shuffled = img_files[:]
    rng.shuffle(shuffled)
    val_names = sorted(f.name for f in shuffled[:int(len(shuffled) * VAL_RATIO)])
    WORK.mkdir(parents=True, exist_ok=True)
    SPLIT_TXT.write_text('\n'.join(f'val\t{n}' for n in val_names), encoding='utf-8')
    print(f"새 split 생성 및 저장: val {len(val_names)}개 → {SPLIT_TXT}")

missing = set(val_names) - {f.name for f in img_files}
assert not missing, f"목록에 있으나 없는 이미지 {len(missing)}개: {sorted(missing)[:3]}"

moved = missing_label = 0
for name in val_names:
    label_path = label_dir / (Path(name).stem + '.txt')
    shutil.move(str(img_dir / name), str(val_img_dir / name))
    if label_path.exists():
        shutil.move(str(label_path), str(val_label_dir / label_path.name))
        moved += 1
    else:
        missing_label += 1
        print(f"라벨 없음: {label_path.name}")

n_train = len([f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS])
n_val   = len([f for f in val_img_dir.iterdir() if f.suffix.lower() in IMG_EXTS])
n_lines = sum(len([l for l in f.read_text().splitlines() if l.strip()])
              for f in val_label_dir.glob('*.txt'))

print(f"train {n_train} / val {n_val} (라벨 {moved}, 누락 {missing_label})")
print(f"val 라벨 줄 수: {n_lines}")

assert n_train + n_val == EXPECTED_TRAIN
assert n_val == len(list(val_label_dir.glob('*.txt'))), "val 이미지/라벨 개수 불일치"
assert n_lines == EXPECTED_VAL_LINES, f"val 라벨 줄 수 불일치: {n_lines} (기대 {EXPECTED_VAL_LINES})"
print("검증 통과")

In [ ]:
import yaml

data_config = {
    'train': '/content/dataset/images/train',
    'val': '/content/dataset/images/val',
    'test': '/content/dataset/images/test',
    'names': ['mandatory', 'prohibitory', 'warning']
}

save_path = '/content/dataset/CCTSDB2021.yaml'

with open(save_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_config, f, default_flow_style=False, allow_unicode=True)

print(f"저장 완료: {save_path}")

# 확인
with open(save_path, 'r') as f:
    print(f.read())

저장 완료: /content/dataset/CCTSDB2021.yaml
names:
- mandatory
- prohibitory
- warning
test: /content/dataset/images/test
train: /content/dataset/images/train
val: /content/dataset/images/val



In [ ]:
import torch

# 이미 패치되어 있으면 다시 패치하지 않도록 가드
if not hasattr(torch, '_true_load_backup'):
    torch._true_load_backup = torch.load  # 진짜 원본을 별도 이름으로 한 번만 백업

def safe_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return torch._true_load_backup(*args, **kwargs)

torch.load = safe_load

In [ ]:
import sys
sys.path.insert(0, '/content/YOLO-TS')
%cd /content/YOLO-TS

from ultralytics import YOLO

/content/YOLO-TS


###Scenario 3

In [ ]:
# model = YOLO('./YOLO-TS_CCTSDB2021.yaml')
# model.load('/content/drive/MyDrive/Generated_TT100K_weather_best.pt')

WARNING ⚠️ no model scale passed. Assuming scale='l'.

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  3     70272  ultralytics.nn.modules.block.C2f             [64, 64, 3, True]             
  2                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  3                  -1  3    279808  ultralytics.nn.modules.block.C2f             [128, 128, 3, True]           
  4                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  5                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  6                  -1  1   1180672  ultralytics.nn.modules.conv.Conv             [256, 512, 3, 2]              
  7                  -1  3   4461

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): C2f(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(160, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (m): ModuleList(
          (0-2): 3 x Bottleneck(
            (cv1): Conv(
              (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
       

In [ ]:
import os, gc, sys, torch
sys.path.insert(0, "/content/YOLO-TS"); os.chdir("/content/YOLO-TS")
gc.collect(); torch.cuda.empty_cache()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

if not hasattr(torch, "_orig_load"):
    torch._orig_load = torch.load
    torch.load = lambda *a, **k: torch._orig_load(*a, **{**k, "weights_only": False})

from ultralytics import YOLO

# ── Scenario 3: Gen-TT → CCTSDB2021 ─────────────────────────────
ARCH_YAML = "/content/YOLO-TS/YOLO-TS_Generated_TT100K_weather.yaml"   # ← 11.03M
TASK1_PT  = "/content/drive/MyDrive/Generated_TT100K_weather_best.pt"
DATA_YAML = "/content/dataset/CCTSDB2021.yaml"
FREEZE    = 17
SEED     = 1
RUN_NAME = f"S3_GenTT_to_CCTSDB_fr{FREEZE}_seed{SEED}"

# ── 1) 체크포인트 자체가 11.03M 구조인지 확인 ───────────────────
src = torch.load(TASK1_PT, map_location="cpu")["model"].state_dict()
print("Task1 params:", sum(v.numel() for v in src.values()))

# ── 2) 모델 생성 + 구조 일치 검증 (불일치면 여기서 중단) ─────────
model = YOLO(ARCH_YAML)
dst = model.model.state_dict()
matched = sum(1 for k, v in src.items() if k in dst and dst[k].shape == v.shape)
print(f"구조 검증: {matched}/{len(dst)}")
assert matched == len(dst), (
    f"구조 불일치 {matched}/{len(dst)} — ARCH_YAML이 Task1과 다릅니다. 학습 중단."
)

model.load(TASK1_PT)      # "Transferred 281/281" 확인

before = {n: b.clone() for n, b in model.model.named_buffers() if 'running_' in n}

# ── 3) 학습 ─────────────────────────────────────────────────────
results = model.train(
    data=DATA_YAML,
    imgsz=640, epochs=150, patience=15, batch=16, nbs=64,
    optimizer="AdamW", lr0=0.0005, lrf=0.01, weight_decay=0.01,
    cos_lr=True, warmup_epochs=5.0,
    mosaic=1.0, close_mosaic=20, mixup=0.15, copy_paste=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=0.0, translate=0.1, scale=0.5, fliplr=0.0, flipud=0.0,
    seed=SEED, deterministic=True, freeze=FREEZE, save_period=10,
    project="/content/drive/MyDrive/runs", name=RUN_NAME,
    amp=True, workers=2, cache=False,
)

changed = [n for n, b in model.model.named_buffers()
           if 'running_' in n and n in before and not torch.equal(before[n], b)]
frozen_changed = [n for n in changed
                  if any(n.startswith(f'model.{i}.') for i in range(FREEZE))]
print(f"BN buffer 변경: 전체 {len(changed)}, frozen 구간 {len(frozen_changed)}")
print(frozen_changed[:5])

WARNING ⚠️ no model scale passed. Assuming scale='l'.

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  3     70272  ultralytics.nn.modules.block.C2f             [64, 64, 3, True]             
  2                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  3                  -1  1    115456  ultralytics.nn.modules.block.C2f             [128, 128, 1, True]           
  4                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  5                  -1  1    460288  ultralytics.nn.modules.block.C2f             [256, 256, 1, True]           
  6                  -1  1   1180672  ultralytics.nn.modules.conv.Conv             [256, 512, 3, 2]              
  7                  -1  1   1838

Task1 params: 11048939
구조 검증: 281/281


Transferred 281/281 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.116 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.0.180 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: task=detect, mode=train, model=/content/YOLO-TS/YOLO-TS_Generated_TT100K_weather.yaml, data=/content/dataset/CCTSDB2021.yaml, epochs=150, patience=15, batch=16, imgsz=640, save=True, save_period=10, cache=False, device=None, workers=2, project=/content/drive/MyDrive/runs, name=GenTT_to_CCTSDB_archfixed_fr17, exist_ok=False, pretrained=False, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=20, resume=False, amp=True, fraction=1.0, profile=False, freeze=17, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, show=False, save_tx

In [ ]:
import sys, os, gc, torch
sys.path.insert(0, "/content/YOLO-TS"); os.chdir("/content/YOLO-TS")

if not hasattr(torch, "_orig_load"):
    torch._orig_load = torch.load
    torch.load = lambda *a, **k: torch._orig_load(*a, **{**k, "weights_only": False})

gc.collect(); torch.cuda.empty_cache()
from ultralytics import YOLO

# BEST_PT = "/content/drive/MyDrive/runs/GenTT_to_CCTSDB_archfixed_fr17/weights/best.pt"
BEST_PT = f"/content/drive/MyDrive/runs/{RUN_NAME}/weights/best.pt"
model = YOLO(BEST_PT)
print("params:", sum(p.numel() for p in model.model.parameters()))   # 11025811이면 정상

metrics = model.val(
    data="/content/dataset/CCTSDB2021.yaml",
    split="test",
    imgsz=640, batch=16, device=0,
    rect=False,          # ← 반드시 유지 (없으면 torch.cat 크기 불일치)
    save_json=True, plots=True,
)

print(metrics.box.map, metrics.box.map50, metrics.box.map75)
print(metrics.box.maps)

Ultralytics YOLOv8.0.180 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO-TS_Generated_TT100K_weather summary (fused): 127 layers, 11016979 parameters, 0 gradients


params: 11025811


val: Scanning /content/dataset/labels/test... 1500 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1500/1500 [00:01<00:00, 1491.82it/s]
val: New cache created: /content/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 94/94 [00:13<00:00,  7.06it/s]
                   all       1500       3228      0.881      0.736      0.796       0.45
             mandatory       1500        718      0.854      0.645      0.704      0.415
           prohibitory       1500       2177       0.92       0.78      0.863      0.501
               warning       1500        333      0.868      0.784      0.822      0.435
Speed: 0.2ms preprocess, 4.0ms inference, 0.0ms loss, 0.7ms postprocess per image
Saving tests/val/predictions.json...
Results saved to tests/val


0.4502485671082969 0.7962146401867586 0.4546871596376885
[     0.4154     0.50072     0.43462]


In [ ]:
# from google.colab import runtime

# runtime.unassign()

In [ ]:
# import shutil
# BEST = f"/content/drive/MyDrive/runs/{RUN_NAME}/weights/best.pt"
# OUT  = f"/content/drive/MyDrive/scenario5_archfixed_fr{FREEZE}.pt"
# shutil.copy2(BEST, OUT)

# t1, t2 = YOLO(TASK1_PT), YOLO(OUT)
# h1, h2 = t1.model.model[-1], t2.model.model[-1]
# print("nc      :", h1.nc, "→", h2.nc)                    # 45 → 3
# print("nl / f  :", (h1.nl, h1.f), (h2.nl, h2.f))         # 같아야 함
# print("cv2 in  :", [b[0].conv.in_channels for b in h1.cv2],
#                    [b[0].conv.in_channels for b in h2.cv2])
# print("params  :", sum(p.numel() for p in t1.model.parameters()),
#                    sum(p.numel() for p in t2.model.parameters()))  # 둘 다 11.03M대
# print("저장:", OUT)

# from google.colab import runtime

# runtime.unassign()